# Task 2 — Profile & Clean

Loads the raw tender records, profiles the data, documents data-quality issues, translates the Arabic text fields to English, classifies each agency into a canonical source and a broader sector, and saves the cleaned result to `data/interim/cleaned.csv`.

## 1. Imports

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
from deep_translator import GoogleTranslator


## 2. Load the latest raw extract

In [ ]:
RAW_DIR = Path("../data/raw")
INTERIM_DIR = Path("../data/interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

latest_file = sorted(RAW_DIR.glob("etimad_all_tenders_*.json"))[-1]

with open(latest_file, encoding="utf-8") as f:
    records = json.load(f)

df = pd.DataFrame(records)
print(f"Loaded {len(df)} rows from {latest_file.name}")


## 3. Profile

For each column: dtype, null count, null percentage, unique count, and a sample value.

In [ ]:
profile = pd.DataFrame({
    "dtype": df.dtypes,
    "null_count": df.isnull().sum(),
    "null_pct": (df.isnull().mean() * 100).round(1),
    "unique_count": df.nunique(),
    "sample_value": df.iloc[0],
})
print(profile)


## 4. Issues List — Task 2

| Issue | Column(s) | Decision |
|---|---|---|
| 100% null | `multipleSearch`, `technicalOrganizationId` | Dropped — no informational value |
| 99%+ null | `agencyCode`, `tenderStatusName` | Dropped — near-empty, redundant with other status fields |
| 55.6% null | `offersOpeningDate` | Kept — still informative for ~44% of rows |
| Same value in every row | `createdAt` | Kept, flagged — likely an extraction timestamp rather than the real tender creation date |


In [ ]:
cols_to_drop = ["multipleSearch", "technicalOrganizationId", "agencyCode", "tenderStatusName"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f"Columns remaining: {len(df.columns)}")


## 5. Translate Arabic text fields to English

Google Translate is used with retries (transient blocks are common with the free tier). Each **unique**
value is translated once and mapped back onto every row that shares it — with 480 rows but far fewer
unique names, this avoids re-sending the same string repeatedly.

A persistent cache (`translation_cache.json`) is loaded and updated so that values translated in a
previous run are never re-sent to the service.

In [ ]:
CACHE_PATH = INTERIM_DIR / "translation_cache.json"

if CACHE_PATH.exists():
    with open(CACHE_PATH, encoding="utf-8") as f:
        translation_cache = json.load(f)
else:
    translation_cache = {}


def translate_text(text, source="ar", target="en"):
    if not text or not isinstance(text, str):
        return text
    for attempt in range(3):
        try:
            result = GoogleTranslator(source=source, target=target).translate(text)
            if result:
                return result
        except Exception:
            time.sleep(5 * (attempt + 1))  # backs off 5s, then 10s, then 15s
    return f"Unknown - translation failed: {text[:30]}"


def save_cache():
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(translation_cache, f, ensure_ascii=False, indent=2)


In [ ]:
COLUMNS_TO_TRANSLATE = ["tenderName", "tenderTypeName", "tenderActivityName", "branchName", "agencyName"]

for col in COLUMNS_TO_TRANSLATE:
    if col not in df.columns:
        continue
    unique_values = df[col].dropna().unique()
    print(f"{col}: {len(unique_values)} unique values (of {len(df)} rows)")

    for i, val in enumerate(unique_values):
        if val not in translation_cache:
            translation_cache[val] = translate_text(val)
            time.sleep(1.5)
        if (i + 1) % 20 == 0:
            print(f"  ... {i + 1}/{len(unique_values)}")

    df[f"{col}_en"] = df[col].map(translation_cache)
    print(f"Done: {col}")

save_cache()
print("Translation complete")


### 5.1 Verify translation success rate

In [ ]:
translated_cols = [f"{c}_en" for c in COLUMNS_TO_TRANSLATE]

for col in translated_cols:
    total = len(df[col])
    failed = df[col].astype(str).str.startswith("Unknown - translation failed").sum()
    empty = df[col].isnull().sum()
    success_rate = round((total - failed - empty) / total * 100, 1)
    print(f"{col}: total={total} | failed={failed} | empty={empty} | success={success_rate}%")


### 5.2 Spot-check: Arabic source vs. English translation

A quick visual sanity check — three real rows per column, original text next to its translation.

In [ ]:
for ar_col in COLUMNS_TO_TRANSLATE:
    en_col = f"{ar_col}_en"
    print(f"=== {ar_col} -> {en_col} ===")
    for i in range(3):
        print(f"  {df[ar_col].iloc[i]}  ->  {df[en_col].iloc[i]}")
    print()


## 6. Classify agency into a canonical source

Every unique `agencyName` value is mapped to itself as the canonical source name — this keeps every
one of the 173 agencies explicitly identified rather than silently grouped or guessed at. Known
branch-level duplicates can be merged later by editing `ENTITY_MAP` directly.

In [ ]:
unique_agencies = df["agencyName"].dropna().unique()
ENTITY_MAP = {agency: agency for agency in unique_agencies}


def map_to_source(agency_name):
    if not agency_name or not isinstance(agency_name, str):
        return "Unknown - extraction issue"
    for known_entity in ENTITY_MAP:
        if agency_name.startswith(known_entity):
            return ENTITY_MAP[known_entity]
    return f"Unmapped - needs review: {agency_name}"


df["source_entity"] = df["agencyName"].apply(map_to_source)

unmapped = df[df["source_entity"].str.startswith("Unmapped")]["source_entity"].unique()
print(f"Distinct sources mapped: {df['source_entity'].nunique()}")
print(f"Agencies still unmapped: {len(unmapped)}")


## 7. Classify into a broader sector

A coarser, keyword-based grouping on top of `source_entity` — useful for reporting sector-level
diversity (security, health, education, ...) rather than raw agency counts.

In [ ]:
SECTOR_KEYWORDS = {
    "Security & Defense": ["أمن", "دفاع", "حرس", "شرطة", "قوات", "عسكري"],
    "Health": ["صحة", "صحية", "مستشفى", "طبي", "طبية"],
    "Education": ["تعليم", "جامعة", "مدرسة", "تدريب", "كلية"],
    "Municipal & Housing": ["بلدية", "أمانة", "إسكان"],
    "Finance & Economy": ["مالية", "اقتصاد", "تجارة", "استثمار", "زكاة", "ضريبة"],
    "Justice": ["عدل", "قضاء", "نيابة", "محكمة"],
    "Transport & Logistics": ["نقل", "طرق", "طيران", "موانئ"],
    "Energy, Water & Environment": ["طاقة", "كهرباء", "مياه", "بيئة", "زراعة", "نفط"],
    "Technology & Data": ["بيانات", "ذكاء اصطناعي", "اتصالات", "تقنية", "معلومات"],
    "Religious & Cultural Affairs": ["أوقاف", "حج", "عمرة", "ثقافة", "إسلامية"],
}


def classify_sector(agency_name):
    if not agency_name or not isinstance(agency_name, str):
        return "Unknown - extraction issue"
    for sector, keywords in SECTOR_KEYWORDS.items():
        if any(kw in agency_name for kw in keywords):
            return sector
    return f"Other - needs review: {agency_name}"


df["sector"] = df["source_entity"].apply(classify_sector)

unclassified = df[df["sector"].str.startswith("Other")]["sector"].unique()
print(f"Sectors used: {df['sector'].nunique()}")
print(f"Agencies not matched to a sector: {len(unclassified)}")


## 8. General cleanup

- Column names standardized to `snake_case`
- Missing values already labeled explicitly at extraction (Task 1) and above
- No further deduplication needed — `tenderId` uniqueness was enforced in Task 1

In [ ]:
df.columns = [
    "".join(["_" + c.lower() if c.isupper() else c for c in col]).lstrip("_")
    for col in df.columns
]
print(df.columns.tolist())


## 9. Save

In [ ]:
output_path = INTERIM_DIR / "cleaned.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Saved to: {output_path}")
